# 02 — Causal feature engineering
Review frozen Phase 3 predictors, exact t+15-minute labels and purged chronological splits. No generation, fitting, cleaning rewrite or dataset export is performed. All records are synthetic.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ml/data/phase3_dataset_manifest.json').is_file())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
approved = Path(os.environ['LOCALAPPDATA']) / 'SmartParking/ml-runtime/Scripts/python.exe'
assert Path(sys.executable).resolve() == approved.resolve()
import numpy as np
import pandas as pd
from ml.generate_data import read_raw
from ml.features import FEATURE_COLUMNS, LAGS, MEAN_WINDOWS, STD_WINDOWS, engineer_features
from ml.prepare_data import read_processed
from ml.train_phase4 import verify_manifest
manifest = json.loads(Path('ml/data/phase3_dataset_manifest.json').read_text(encoding='utf-8'))
verify_manifest(manifest)
raw = read_raw(manifest['raw_file']['path'])
frame = read_processed(manifest['processed_file']['path'], manifest['schema'])
print('Interpreter:', sys.executable)
print('Rows:', len(frame), 'Predictors:', len(FEATURE_COLUMNS))

Interpreter: C:\Users\vighn\AppData\Local\SmartParking\ml-runtime\Scripts\python.exe
Rows: 70515 Predictors: 37


## Predictor availability
37 predictors contain current states/totals/transitions, local calendar cycles, lags of 1/5/15/30/60/1440/10080 minutes, past-only rolling means/std and current-minus-lag-15 change. Current measurements must already be available at inference time.
Rolling windows are [t−w,t), using `shift(1)` and population std (`ddof=0`). Target, target timestamp, split and provenance are excluded from predictors. Label y(t)=total(t+15); it is joined by exact timestamp, never row position after partitioning.

In [2]:
assert list(FEATURE_COLUMNS) == manifest['features'] and len(FEATURE_COLUMNS) == 37
assert not set(FEATURE_COLUMNS).intersection(manifest['non_predictor_columns'])
assert len(frame) == manifest['processed_rows'] and not frame.isna().any().any()
assert (frame.target_timestamp - frame.timestamp).eq(pd.Timedelta(minutes=15)).all()
timeline = raw.set_index('timestamp').total_occupied
expected = timeline.reindex(pd.DatetimeIndex(frame.target_timestamp)).to_numpy()
assert np.array_equal(expected, frame.occupied_15min_ahead.to_numpy())
assert frame.data_source.eq('synthetic').all()
display(frame[['timestamp', 'total_occupied', 'target_timestamp', 'occupied_15min_ahead', 'split']].head(4))
print('PASS: every frozen target equals occupancy at the exact future timestamp')

,timestamp,total_occupied,target_timestamp,occupied_15min_ahead,split
0,2026-09-21 00:00:00+05:30,0,2026-09-21 00:15:00+05:30,0,train
1,2026-09-21 00:01:00+05:30,0,2026-09-21 00:16:00+05:30,0,train
2,2026-09-21 00:02:00+05:30,0,2026-09-21 00:17:00+05:30,0,train
3,2026-09-21 00:03:00+05:30,0,2026-09-21 00:18:00+05:30,0,train


PASS: every frozen target equals occupancy at the exact future timestamp


In [3]:
origin_index = pd.Index(raw.timestamp).get_indexer(frame.timestamp)
assert (origin_index >= 10080).all()
total = raw.total_occupied.astype('float64')
for lag in LAGS:
    np.testing.assert_allclose(frame[f'occupied_lag_{lag}'], total.shift(lag).iloc[origin_index], rtol=0, atol=1e-6)
for window in MEAN_WINDOWS:
    np.testing.assert_allclose(frame[f'occupied_mean_{window}'], total.shift(1).rolling(window).mean().iloc[origin_index], rtol=0, atol=1e-6)
for window in STD_WINDOWS:
    np.testing.assert_allclose(frame[f'occupied_std_{window}'], total.shift(1).rolling(window).std(ddof=0).iloc[origin_index], rtol=0, atol=1e-6)
np.testing.assert_allclose(frame.occupied_change_15, (total - total.shift(15)).iloc[origin_index], rtol=0, atol=1e-6)
print('PASS: all persisted lag, rolling and change values match causal calculations')

PASS: all persisted lag, rolling and change values match causal calculations


## Future intervention and split checks
Perturb only future observations in a small in-memory prefix, then recompute the existing feature function. Every predictor at and before the cutoff must remain unchanged. Frozen files are not touched. Boundary checks require each preceding partition's final target to be earlier than the next partition's first origin.

In [4]:
prefix = raw.iloc[:10200].copy()
cutoff = 10100
before = engineer_features(prefix)
changed = prefix.copy()
changed.loc[changed.index > cutoff, 'total_occupied'] = 12 - changed.loc[changed.index > cutoff, 'total_occupied']
after = engineer_features(changed)
pd.testing.assert_frame_equal(before.loc[:cutoff, list(FEATURE_COLUMNS)], after.loc[:cutoff, list(FEATURE_COLUMNS)])
boundaries = []
for name, expected_boundary in manifest['split_boundaries'].items():
    part = frame.loc[frame.split.eq(name)]
    assert len(part) == expected_boundary['rows']
    assert part.timestamp.is_unique and part.timestamp.is_monotonic_increasing
    assert part.timestamp.diff().iloc[1:].eq(pd.Timedelta(minutes=1)).all()
    boundaries.append({'split': name, 'rows': len(part), 'first_origin': part.timestamp.iloc[0], 'last_origin': part.timestamp.iloc[-1], 'last_target': part.target_timestamp.iloc[-1]})
for left, right in zip(boundaries, boundaries[1:]):
    assert left['last_target'] < right['first_origin']
    assert right['first_origin'] - left['last_origin'] == pd.Timedelta(minutes=16)
display(pd.DataFrame(boundaries))
print('PASS: future-intervention invariance and both 15-row purge gaps')

,split,rows,first_origin,last_origin,last_target
0,train,49360,2026-09-21 00:00:00+05:30,2026-10-25 06:39:00+05:30,2026-10-25 06:54:00+05:30
1,validation,10577,2026-10-25 06:55:00+05:30,2026-11-01 15:11:00+05:30,2026-11-01 15:26:00+05:30
2,test,10578,2026-11-01 15:27:00+05:30,2026-11-08 23:44:00+05:30,2026-11-08 23:59:00+05:30


PASS: future-intervention invariance and both 15-row purge gaps


## Interpretation
Feature construction is deterministic and does not fit statistics across partitions. Training/validation/test sizes are 49,360/10,577/10,578 after a seven-day warmup, 15-minute tail exclusion and two boundary purges. These checks establish temporal correctness of synthetic artifacts, not real-world predictive validity.